In [1]:
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier

# 글로벌 시드 고정
np.random.seed(42)

def run_complexity_experiment():
    print("-" * 60)
    print("[실험 2 & 4 & 5] 샘플 수와 특성 수 변화에 따른 복잡도 격차 추적")
    print("-" * 60)

    # ----------------------------------------------------------------------
    # 시나리오 A: 샘플 수(m) 폭발 (m >> n) -> 실험 2(시간복잡도) & 실험 4(손실함수 비교)
    # ----------------------------------------------------------------------
    X_large_m, y_large_m = make_classification(n_samples=40000, n_features=10, random_state=42)

    models_m = {
        "SVC (Linear, Dual=True)": SVC(kernel="linear"),
        "LinearSVC (Squared Hinge, Dual=False)": LinearSVC(loss="squared_hinge", dual=False, random_state=42, max_iter=10000),
        "LinearSVC (Hinge, Dual=True)": LinearSVC(loss="hinge", dual=True, random_state=42, max_iter=10000),
        "SGDClassifier (Hinge)": SGDClassifier(loss="hinge", random_state=42)
    }

    print("\n▶ 시나리오 A: 샘플 수 대량 증가 (m = 40,000, n = 10)")
    for name, model in models_m.items():
        # SVC(libsvm)는 O(m^2) 병목으로 시간이 폭발하므로 일부만 샘플링하여 유령 연산 실측
        if "SVC" in name:
            start = time.time()
            model.fit(X_large_m[:8000], y_large_m[:8000]) # 8천 개만 학습해도 지연 발생
            end = time.time()
            print(f" └ {name} [주의: 8,000개 샘플만 훈련]: {end - start:.4f}초")
        else:
            start = time.time()
            model.fit(X_large_m, y_large_m)
            end = time.time()
            print(f" └ {name} [40,000개 전수 훈련]: {end - start:.4f}초")

    # ----------------------------------------------------------------------
    # 시나리오 B: 특성 수(n) 폭발 (n >> m) -> 실험 5(원문제 vs 쌍대문제 수렴 속도 격차)
    # ----------------------------------------------------------------------
    X_large_n, y_large_n = make_classification(n_samples=800, n_features=5000, random_state=42)

    print("\n▶ 시나리오 B: 특성(Feature) 수 대량 증가 (m = 800, n = 5,000)")

    # 5번 주제 검증: n >> m 일 때 원문제(Dual=False)와 쌍대문제(Dual=True)의 차원 역전 및 수속 속도 우회 증명
    start = time.time()
    LinearSVC(dual=False, max_iter=10000, random_state=42).fit(X_large_n, y_large_n)
    print(f" └ LinearSVC 원문제 모드 (dual=False, 가중치 w 타겟): {time.time() - start:.4f}초")

    start = time.time()
    LinearSVC(dual=True, max_iter=10000, random_state=42).fit(X_large_n, y_large_n)
    print(f" └ LinearSVC 쌍대문제 모드 (dual=True, 승수 alpha 타겟): {time.time() - start:.4f}초")

# 실험 실행
if __name__ == "__main__":
    run_complexity_experiment()

------------------------------------------------------------
[실험 2 & 4 & 5] 샘플 수와 특성 수 변화에 따른 복잡도 격차 추적
------------------------------------------------------------

▶ 시나리오 A: 샘플 수 대량 증가 (m = 40,000, n = 10)
 └ SVC (Linear, Dual=True) [주의: 8,000개 샘플만 훈련]: 2.2711초
 └ LinearSVC (Squared Hinge, Dual=False) [주의: 8,000개 샘플만 훈련]: 0.0108초
 └ LinearSVC (Hinge, Dual=True) [주의: 8,000개 샘플만 훈련]: 0.0593초


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


 └ SGDClassifier (Hinge) [40,000개 전수 훈련]: 0.5012초

▶ 시나리오 B: 특성(Feature) 수 대량 증가 (m = 800, n = 5,000)
 └ LinearSVC 원문제 모드 (dual=False, 가중치 w 타겟): 4.3515초
 └ LinearSVC 쌍대문제 모드 (dual=True, 승수 alpha 타겟): 0.1454초
